In [20]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer

nltk.download('stopwords')
nltk.download('punkt')

df = pd.read_excel('./data/corpus/text_corpus_news.xlsx')

# Удаление шума
df['label'] = (
    df[['Increase', 'Stable', 'Degrease']]
    .notna()
    .idxmax(axis=1)
    .map({'Increase': 0, 'Stable': 1, 'Degrease': 2})
)

df = df.drop(['Unnamed: 0','Date','Increase', 'Stable', 'Degrease'], axis=1)
df = df.sample(frac=1.0, random_state=42)
df = df.dropna()

pattern = r"Investing\.com.*?Я согласен"
df['Text'] = (
    df['Text']
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(pattern, '', regex=True, flags=re.S)
        .str.replace(r'\n+', '\n', regex=True)
        .str.strip()
)

# Стемминг + стоп-слова
russian_stopwords = set(stopwords.words('russian'))
stemmer = SnowballStemmer("russian")

def preprocess_text(text):
    tokens = word_tokenize(text, language='russian')
    stems = []
    for token in tokens:
        if token.isalpha():
            stem = stemmer.stem(token.lower())
            if stem not in russian_stopwords:
                stems.append(stem)
    return ' '.join(stems)

df['Text'] = df['Text'].apply(preprocess_text)
df = df[df['Text'].str.strip() != '']

df = df.rename(columns={"label": "labels", "Text": "text"})
df = df.dropna()

df.to_csv('data/test_data/processed_data.csv', index=False)
print("Пример результата:")
print(df.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Пример результата:
                                                    text  labels
4468   индекс мосбирж ртс конц торг остава минус ввид...       2
17521  итог четверг доллар прибав копейк евр подорожа...       2
28516  сенсац прошл недел стал решен обязательн прода...       1
26264  напомн наш цб собира внепланов заседан ма опус...       1
6689   курс доллар установлен цб рф июн повыш курс ев...       2
